# YouTube Transcript RAG

Ask questions about any YouTube video using Retrieval-Augmented Generation.

## 1. Imports & Configuration

In [ ]:
import os
import re
from urllib.parse import urlparse, parse_qs
from dotenv import load_dotenv

from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled, NoTranscriptFound
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEndpoint
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
HF_TOKEN = os.getenv("HUGGINGFACEHUB_API_TOKEN")

assert OPENAI_API_KEY, "Set OPENAI_API_KEY in .venv"
assert HF_TOKEN, "Set HUGGINGFACEHUB_API_TOKEN in .venv"

print("Keys loaded successfully.")

## 2. Transcript Extraction

In [ ]:
def extract_video_id(url: str) -> str:
    """Parse a YouTube URL (any format) and return the video ID."""
    parsed = urlparse(url)
    if parsed.hostname in ("youtu.be",):
        return parsed.path.lstrip("/")
    if parsed.hostname in ("www.youtube.com", "youtube.com"):
        if parsed.path == "/watch":
            return parse_qs(parsed.query)["v"][0]
        if parsed.path.startswith("/embed/"):
            return parsed.path.split("/")[2]
        if parsed.path.startswith("/shorts/"):
            return parsed.path.split("/")[2]
    raise ValueError(f"Could not extract video ID from: {url}")


def get_transcript(youtube_url: str, languages: list = None) -> str:
    """Fetch and concatenate the transcript for a YouTube video."""
    video_id = extract_video_id(youtube_url)
    languages = languages or ["en"]
    try:
        transcript_list = YouTubeTranscriptApi.get_transcript(video_id, languages=languages)
    except NoTranscriptFound:
        # fallback: grab whatever language is available
        transcripts = YouTubeTranscriptApi.list_transcripts(video_id)
        transcript_list = transcripts.find_generated_transcript(
            transcripts._generated_transcripts.keys()
        ).fetch()
    
    text = " ".join(item["text"] for item in transcript_list)
    return text


# 3Blue1Brown: "But what is a neural network?"
DEMO_URL = "https://www.youtube.com/watch?v=aircAruvnKk"

transcript = get_transcript(DEMO_URL)
print(f"Transcript length : {len(transcript):,} characters")
print(f"Preview           : {transcript[:400]}...")

## 3. Chunking

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = text_splitter.create_documents(
    [transcript],
    metadatas=[{"source": DEMO_URL}] * 1  # will be spread per chunk below
)

# Attach source metadata to every chunk
for chunk in chunks:
    chunk.metadata["source"] = DEMO_URL

print(f"Number of chunks : {len(chunks)}")
print(f"\nSample chunk:\n{chunks[0].page_content}")

## 4. Embeddings & Vector Store (FAISS)

In [ ]:
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=OPENAI_API_KEY,
)

vectorstore = FAISS.from_documents(chunks, embeddings)

# Persist so we don't re-embed on every run
vectorstore.save_local("faiss_index")
print(f"Vector store saved. Total vectors: {vectorstore.index.ntotal}")

## 5. HuggingFace LLM (Mistral-7B-Instruct)

In [ ]:
llm = HuggingFaceEndpoint(
    repo_id="mistralai/Mistral-7B-Instruct-v0.3",
    max_new_tokens=512,
    temperature=0.1,
    huggingfacehub_api_token=HF_TOKEN,
)

# Quick sanity check
print(llm.invoke("Say 'RAG pipeline ready' and nothing else."))

## 6. RAG Chain

In [ ]:
PROMPT_TEMPLATE = """You are an assistant that answers questions based on a YouTube video transcript.
Use only the context provided. If the answer is not in the context, say "I couldn't find that in the video."

Context:
{context}

Question: {question}

Answer:"""

prompt = PromptTemplate(
    template=PROMPT_TEMPLATE,
    input_variables=["context", "question"],
)

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5},
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    chain_type_kwargs={"prompt": prompt},
    return_source_documents=True,
)

print("RAG chain ready.")

## 7. Ask Questions

In [ ]:
def ask(question: str):
    result = qa_chain.invoke({"query": question})
    print(f"Q: {question}")
    print(f"\nA: {result['result'].strip()}")
    print("\n--- Retrieved chunks ---")
    for i, doc in enumerate(result["source_documents"], 1):
        print(f"[{i}] {doc.page_content[:180]}...")
    print()
    return result

ask("What is the main topic of this video?")

In [ ]:
ask("How are neurons in a neural network structured?")

In [ ]:
ask("What analogy does the speaker use to explain neural networks?")

## 8. Multi-Video RAG

Extend the same pipeline to multiple YouTube videos and keep track of which video each answer came from.

In [ ]:
def build_multi_video_rag(url_list: list[str]) -> FAISS:
    """Build a FAISS vector store from multiple YouTube URLs."""
    all_chunks = []
    for url in url_list:
        try:
            print(f"Fetching transcript: {url}")
            text = get_transcript(url)
            chunks = text_splitter.create_documents([text])
            for chunk in chunks:
                chunk.metadata["source"] = url
            all_chunks.extend(chunks)
            print(f"  -> {len(chunks)} chunks")
        except (TranscriptsDisabled, NoTranscriptFound) as e:
            print(f"  -> Skipped ({e})")
    
    store = FAISS.from_documents(all_chunks, embeddings)
    store.save_local("faiss_multi_index")
    print(f"\nTotal vectors: {store.index.ntotal}")
    return store


# Example: three 3Blue1Brown neural network videos
VIDEO_URLS = [
    "https://www.youtube.com/watch?v=aircAruvnKk",  # What is a neural network?
    "https://www.youtube.com/watch?v=IHZwWFHWa-w",  # Gradient descent
    "https://www.youtube.com/watch?v=Ilg3gGewQ5U",  # Backpropagation
]

multi_store = build_multi_video_rag(VIDEO_URLS)

In [ ]:
# Rebuild the chain using the multi-video store
multi_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=multi_store.as_retriever(search_kwargs={"k": 5}),
    chain_type_kwargs={"prompt": prompt},
    return_source_documents=True,
)

result = multi_chain.invoke({"query": "What is gradient descent and why is it important?"})
print(result["result"].strip())
print("\nSources:")
for doc in result["source_documents"]:
    print(f"  {doc.metadata['source']}")

## 9. Load Saved Index (Skip Re-embedding)

Reload a previously saved FAISS index without calling the OpenAI embeddings API again.

In [ ]:
loaded_store = FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True,
)
print(f"Loaded {loaded_store.index.ntotal} vectors from disk.")